In [45]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Load the dataset
data = pd.read_csv("mentalhealthdataset.csv")

# Clean and handle missing values
data = data.drop_duplicates()
data['self_employed'] = data['self_employed'].fillna('No')
data = data.drop(columns=['Timestamp'])

In [47]:
# --- Feature Mapping for Risk Function Logic (and all other features) ---

# Map binary/ordinal columns to numeric/simplified values 
data['treatment'] = data['treatment'].map({'No': 0, 'Yes': 1})
data['Mental_Health_History'] = data['Mental_Health_History'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['Mood_Swings'] = data['Mood_Swings'].map({'Low': 0, 'Medium': 1, 'High': 2})

# Define the risk_level function
def define_risk(row):
    if (row['treatment'] == 1 or \
        (row['Mental_Health_History'] == 2 and row['Mood_Swings'] == 2)):
        return 'High'
    if (row['treatment'] == 0 and \
        row['Mental_Health_History'] == 0 and \
        row['Mood_Swings'] == 0):
        return 'Low'
    return 'Medium'

data['Risk_Level'] = data.apply(define_risk, axis=1)

# Map the rest of the ordinal columns to integers
data['Days_Indoors'] = data['Days_Indoors'].map({'Go out Every day': 0, '1-14 days': 1, '15-30 days': 2, '31-60 days': 3, 'More than 2 months': 4})
data['Growing_Stress'] = data['Growing_Stress'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['Changes_Habits'] = data['Changes_Habits'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['Work_Interest'] = data['Work_Interest'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['Social_Weakness'] = data['Social_Weakness'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['mental_health_interview'] = data['mental_health_interview'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['care_options'] = data['care_options'].map({'No': 0, 'Maybe': 1, 'Yes': 2, 'Not sure': 1})

# Apply Label Encoding to the remaining binary-like categorical columns 
le_cols=['Gender','self_employed','family_history','Coping_Struggles']
for col in le_cols:
    data[col]=LabelEncoder().fit_transform(data[col])
    
print("Risk Level Distribution:")
print(data['Risk_Level'].value_counts())

Risk Level Distribution:
Risk_Level
High      160276
Medium    112509
Low        17266
Name: count, dtype: int64


In [49]:
# --- Full Numerical Encoding for SMOTE ---
# One-Hot Encode nominal categorical features: Country and Occupation
data = pd.get_dummies(data, columns=['Country', 'Occupation'], drop_first=True)

# Convert all boolean columns (created by pd.get_dummies) to integers (0 or 1)
bool_cols = data.select_dtypes('bool').columns
data[bool_cols] = data[bool_cols].astype(int)


# --- Define Features (X) and Target (y) ---

# Drop the columns used to create the target, and the target itself
X = data.drop(columns=['Risk_Level', 'treatment', 'Mental_Health_History', 'Mood_Swings'])
y = data['Risk_Level']

# --- Split the Data ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [51]:
# --- SMOTE SAMPLING STRATEGY and Application ---
majority_count = y_train.value_counts()['High']
low_count = y_train.value_counts()['Low']

sampling_strategy = {
    'Low': low_count * 4,
    'Medium': majority_count,
    'High': majority_count
}

smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42)
# X is now fully numerical, so SMOTE will run without the ValueError
X_res, y_res = smote.fit_resample(X_train, y_train)

print("Original training distribution:\n", y_train.value_counts())
print("Resampled training distribution:\n", y_res.value_counts())

# CatBoost needs the target (y) mapped to integers (0, 1, 2)
label_map = {'Low': 0, 'Medium': 1, 'High': 2}
y_train_catboost = y_res.map(label_map)
y_test_catboost = y_test.map(label_map)

C:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Original training distribution:
 Risk_Level
High      112193
Medium     78756
Low        12086
Name: count, dtype: int64
Resampled training distribution:
 Risk_Level
High      112193
Medium    112193
Low        48344
Name: count, dtype: int64


In [53]:
# --- Identify Categorical Features for CatBoost (by index in the final X) ---
# We pass the indices of features that originated from categorical data and are not OHE.
categorical_cols_to_treat_as_categorical = [
    'Gender', 'self_employed', 'family_history', 'Coping_Struggles', 
    'Days_Indoors', 'Growing_Stress', 'Changes_Habits', 
    'Work_Interest', 'Social_Weakness', 'mental_health_interview', 
    'care_options'
]

# Get the indices for the CatBoost model
cat_features_indices = [X.columns.get_loc(col) for col in categorical_cols_to_treat_as_categorical if col in X.columns]


# --- CatBoost Model Training and Evaluation ---

catboost_model = CatBoostClassifier(
    iterations=400,          
    depth=10,                 
    learning_rate=0.05,       
    # Pass the indices of the categorical features.
    cat_features=cat_features_indices,  
    random_state=42,          
    verbose=200,              
    objective='MultiClass',
    # SMOTE balanced the training set
    class_weights=[1.0, 1.0, 1.0] 
)

print("\nFitting CatBoost Model on SMOTE Resampled Data...")
catboost_model.fit(X_res, y_train_catboost)


y_pred_catboost_int = catboost_model.predict(X_test)
y_pred_catboost_flattened = y_pred_catboost_int.flatten()
reverse_label_map = {0: 'Low', 1: 'Medium', 2: 'High'}
y_pred_catboost_labels = pd.Series(y_pred_catboost_flattened).map(reverse_label_map)


print("\n--- CatBoost Classification Report (SMOTE with Cat Features) ---\n", classification_report(y_test, y_pred_catboost_labels))
print("CatBoost Confusion Matrix:\n", confusion_matrix(y_test, y_pred_catboost_labels))
print("\nWeighted Average F1-score:", classification_report(y_test, y_pred_catboost_labels, output_dict=True)['weighted avg']['f1-score'])


Fitting CatBoost Model on SMOTE Resampled Data...
0:	learn: 1.0768269	total: 1.35s	remaining: 8m 57s
200:	learn: 0.4960724	total: 3m 50s	remaining: 3m 47s
399:	learn: 0.4373108	total: 7m 46s	remaining: 0us

--- CatBoost Classification Report (SMOTE with Cat Features) ---
               precision    recall  f1-score   support

        High       0.84      0.71      0.77     48083
         Low       0.59      0.93      0.72      5180
      Medium       0.71      0.80      0.75     33753

    accuracy                           0.76     87016
   macro avg       0.71      0.81      0.75     87016
weighted avg       0.77      0.76      0.76     87016

CatBoost Confusion Matrix:
 [[34077  2968 11038]
 [  352  4819     9]
 [ 6263   354 27136]]

Weighted Average F1-score: 0.7599386494594114
